In [1]:
# Import libraries
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import time

from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import minimize
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
from joblib import Parallel, delayed


In [2]:
# Define parameters

N = 6          # number of qubits
J = 1.0        # coupling strength
reps = 2
tol = 0.1 
epsilon_N = 1e-6 * (4 / N) ** 4    # symmetry-breaking field

# Define h values
h_values = np.concatenate([
    np.linspace(0.2, 0.8, 8),      # Ordered region 8 points
    np.linspace(0.8, 1.2, 10),     # Critical region 10 points 
    np.linspace(1.2, 3.0, 7)       # Disordered region 7 points
])    


# Finite-size scaling parameters (exact values for 1D TFIM)
N_values = [4, 6, 8, 10]    # system sizes to sweep
n_restarts = 1              # number of restarts per point
num_processes = 4           # number of processes for parallelization
beta_exp   = 0.125
nu_exp     = 1.0

In [3]:
# LiHoF4 physical parameters
# Source: Bitko et al. PRL 77, 940 (1996)

mu_B   = 5.788e-2    # meV / Tesla
k_B    = 8.617e-2    # meV / K

# Experimentally measured values
B_c    = 4.9         # Tesla — critical field at T=0
T_c    = 1.53        # K — critical temperature at B=0

# Convert critical temperature to energy scale
J_meV  = k_B * T_c / 2    # factor of 2 from mean-field relation T_c = 2J/k_B
h_c_meV = J_meV            # at the critical point h_c = J by definition

print(f"J     = {J_meV:.4f} meV")
print(f"h_c   = {h_c_meV:.4f} meV")
print(f"h_c/J = {h_c_meV/J_meV:.4f}  (should be 1.0)")

# Map applied field B to effective transverse field h
# Calibrated so that B = B_c maps to h = J
def field_to_h(B_tesla):
    return h_c_meV * (B_tesla / B_c)

# Sweep in physical units
B_values      = np.linspace(0.5, 9.0, 25)
h_values_phys = np.array([field_to_h(B) for B in B_values])

print(f"\nField sweep: {B_values[0]:.1f} T → {B_values[-1]:.1f} T")
print(f"h sweep:     {h_values_phys[0]:.4f} → {h_values_phys[-1]:.4f} meV")
print(f"h/J sweep:   {h_values_phys[0]/J_meV:.2f} → {h_values_phys[-1]/J_meV:.2f}")

J     = 0.0659 meV
h_c   = 0.0659 meV
h_c/J = 1.0000  (should be 1.0)

Field sweep: 0.5 T → 9.0 T
h sweep:     0.0067 → 0.1211 meV
h/J sweep:   0.10 → 1.84


In [ ]:
# Build the Transverse Field Ising Model Hamiltonian
def build_tfim_hamiltonian (N: int, J: float, h: float, epsilon: float = 1e-6) -> sp.csr_matrix:
    """
    Build the TFIM Hamiltonian as a sparse CSR matrix in the Z-basis.
    H = -J Σ Z_i Z_{i+1}  -  h Σ X_i  -  ε Σ Z_i
    """
    dim = 1 << N          # 2^N
    diag = np.zeros(dim)
    rows, cols, data = [], [], []
 
    for state in range(dim):
        # ── ZZ terms (diagonal) ──────────────────────────────────────────────
        for i in range(N - 1):
            zi = 1 - 2 * ((state >> i) & 1)       # +1 or -1
            zj = 1 - 2 * ((state >> (i + 1)) & 1)
            diag[state] -= J * zi * zj
 
        # ── ε Z terms (diagonal) ────────────────────────────────────────────
        for i in range(N):
            zi = 1 - 2 * ((state >> i) & 1)
            diag[state] -= epsilon * zi
 
        # ── h X terms (off-diagonal: flip bit i) ────────────────────────────
        for i in range(N):
            flipped = state ^ (1 << i)
            rows.append(state)
            cols.append(flipped)
            data.append(-h)
 
    H = sp.diags(diag, format="csr") + sp.csr_matrix(
        (data, (rows, cols)), shape=(dim, dim)
    )
    return H
 
 
def exact_ground_state_energy(N, J, h):
    H = build_tfim_hamiltonian(N, J, h)
    # eigsh is much faster than eigvalsh for large sparse matrices; k=1 suffices here
    vals = spla.eigsh(H, k=2, which="SA", return_eigenvectors=False)
    return float(vals[0])
 
 
def exact_magnetisation(N: int, J: float, h: float) -> float:
    epsilon_map   = {4: 1e-6, 6: 1e-4, 8: 1e-4, 10: 1e-4}
    epsilon_local = epsilon_map.get(N, 1e-4)

    H_matrix             = np.array(build_tfim_hamiltonian(N, J, h,
                                     epsilon=epsilon_local).to_matrix())
    eigenvalues, eigvecs = np.linalg.eigh(H_matrix)
    ground_state         = np.ascontiguousarray(eigvecs[:, 0])
    M_matrix             = np.array(build_magnetisation_op(N).to_matrix())
    M_val                = ground_state.conj() @ M_matrix @ ground_state
    return abs(float(np.real(M_val)))
 
 
def exact_energy_gap(N, J, h):
    H = build_tfim_hamiltonian(N, J, h)
    vals = spla.eigsh(H, k=2, which="SA", return_eigenvectors=False)
    return float(np.sort(vals)[1] - np.sort(vals)[0])

def build_ansatz(N: int, reps: int = 2):
    return efficient_su2(num_qubits=N, reps=reps, entanglement="linear")
 
 
def make_vqe_cost(ansatz, H):
    """
    Returns a cost function that takes a parameter vector and returns ⟨H⟩.
    Closure over a fixed ansatz and sparse Hamiltonian.
    """
    def cost_fn(params):
        sv = Statevector(ansatz.assign_parameters(params))
        psi = sv.data          # complex numpy array, length 2^N
        # Sparse expectation value: ⟨ψ|H|ψ⟩
        return float(np.real(psi.conj() @ H.dot(psi)))
    return cost_fn
 
 
def run_vqe_optimised(N, J, h, reps=2, seed=42, warm_start_params=None):
    H = build_tfim_hamiltonian(N, J, h)
    ansatz   = build_ansatz(N, reps)
    cost_fn  = make_vqe_cost(ansatz, H)
 
    near_critical = abs(h / J - 1.0) < 0.3
    ftol    = 1e-9 if near_critical else 1e-6
    gtol    = 1e-6 if near_critical else 1e-5
    maxiter = 500  if near_critical else 300
 
    if warm_start_params is not None:
        starting_points = [warm_start_params]
    else:
        attempts = n_restarts if (near_critical and warm_start_params is None) else 1
        rng = np.random.default_rng(seed)
        starting_points = [
            rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
            for _ in range(attempts)
        ]
 
    best_energy, best_params = np.inf, None
    for p0 in starting_points:
        res = minimize(cost_fn, p0, method="L-BFGS-B",
                       options={"maxiter": maxiter, "ftol": ftol, "gtol": gtol})
        if res.fun < best_energy:
            best_energy, best_params = res.fun, res.x
 
    return best_energy, best_params
 
 
def magnetisation_from_params(N, ansatz, params):
    sv  = Statevector(ansatz.assign_parameters(params)).data
    M   = 0.0
    for i in range(N):
        for state in range(1 << N):
            zi = 1 - 2 * ((state >> i) & 1)
            M += zi * (abs(sv[state]) ** 2)
    return abs(M / N)

# Parallelise the h sweep for exact computations

def _exact_single_h(h, N, J):
    """Compute all exact quantities for one (N, h) pair."""
    H = build_tfim_hamiltonian(N, J, h)
    vals, vecs = spla.eigsh(H, k=2, which="SA")
    idx  = np.argsort(vals)
    gs   = vecs[:, idx[0]]
    gap  = float(vals[idx[1]] - vals[idx[0]])
    energy = float(vals[idx[0]])
 
    # Magnetisation
    M = 0.0
    for i in range(N):
        for state in range(1 << N):
            zi = 1 - 2 * ((state >> i) & 1)
            M += zi * gs[state] ** 2
    mag = abs(M / N)
    return energy, mag, gap
 
 
def exact_benchmark_sweep_parallel(N_values, J, h_values, n_jobs=-1):
    benchmark = {}
    for N in N_values:
        print(f"Benchmarking N={N} (parallel over h)...")
        results_h = Parallel(n_jobs=n_jobs)(
            delayed(_exact_single_h)(h, N, J) for h in h_values
        )
        energies, mags, gaps = zip(*results_h)
        benchmark[N] = {
            "energies"      : np.array(energies),
            "magnetisations": np.array(mags),
            "gaps"          : np.array(gaps),
        }
    return benchmark

# Parallelise over N
def classify_phase(h, J, tol):
    ratio = h / J
    if abs(ratio - 1.0) < tol: return " CRITICAL "
    elif ratio < 1.0:           return " ORDERED  "
    else:                       return "DISORDERED"
 
 
def sweep_phase_diagram_optimised(N, J, h_values, reps=2):
    """Sequential sweep over h with warm-starting."""
    n_reps = reps if N <= 6 else reps + 1
    vqe_energies, exact_energies, magnetisations = [], [], []
    prev_params = None
 
    for i, h in enumerate(h_values):
        phase = classify_phase(h, J, tol)
        print(f"N={N} [{i+1}/{len(h_values)}]  h/J={h/J:.3f}  [{phase}]", end="  ")
 
        E_exact = exact_ground_state_energy(N, J, h)
        exact_energies.append(E_exact)
 
        E_vqe, params = run_vqe_optimised(N, J, h, reps=n_reps,
                                           warm_start_params=prev_params)
        prev_params = params
        vqe_energies.append(E_vqe)
 
        ansatz = build_ansatz(N, n_reps)
        M = magnetisation_from_params(N, ansatz, params)
        magnetisations.append(M)
 
        print(f"E_exact={E_exact:.4f}  E_vqe={E_vqe:.4f}  "
              f"err={abs(E_vqe-E_exact):.4f}  |M|={M:.4f}")
 
    return (np.array(vqe_energies),
            np.array(exact_energies),
            np.array(magnetisations))
 
 
def _run_single_N(N, J, h_values, reps):
    """Worker: full VQE sweep for one system size."""
    n_reps = reps if N <= 6 else reps + 1
    print(f"\n{'='*50}\nRunning sweep for N={N}, reps={n_reps}\n{'='*50}")
    t0 = time.time()
    vqe_e, exact_e, mags = sweep_phase_diagram_optimised(N, J, h_values, reps)
    print(f"Time for N={N}: {time.time()-t0:.1f}s")
    return N, vqe_e, exact_e, mags
 
 
def run_finite_size_sweep_parallel(N_values, J, h_values, reps=2, n_jobs=4):
    """
    Run all N sweeps in parallel.
    NOTE: each VQE sweep is still sequential over h (warm-starting).
    Parallelism is across N values — the main bottleneck.
    """
    raw = Parallel(n_jobs=n_jobs, prefer="threads")(

        delayed(_run_single_N)(N, J, h_values, reps)
        for N in N_values
    )
    return {N: {"vqe_energies": ve, "exact_energies": ee, "magnetisations": m}
            for N, ve, ee, m in raw}

if __name__ == "__main__":
    t_start = time.time()
 
    # All N sweeps in parallel
    results = run_finite_size_sweep_parallel(N_values, J, h_values, reps=reps,
                                              n_jobs=num_processes)

    benchmark = exact_benchmark_sweep_parallel(N_values, J, h_values, n_jobs=-1)
 
    print(f"\nTotal wall time: {time.time()-t_start:.1f}s")
 



Running sweep for N=10, reps=3
N=10 [1/25]  h/J=0.200  [ ORDERED  ]  
Running sweep for N=6, reps=2
N=6 [1/25]  h/J=0.200  [ ORDERED  ]  
Running sweep for N=4, reps=2
N=4 [1/25]  h/J=0.200  [ ORDERED  ]  
Running sweep for N=8, reps=3
N=8 [1/25]  h/J=0.200  [ ORDERED  ]  E_exact=-3.0587  E_vqe=-3.0599  err=0.0012  |M|=0.9876
N=4 [2/25]  h/J=0.286  [ ORDERED  ]  E_exact=-5.0802  E_vqe=-5.0801  err=0.0001  |M|=0.9898
N=6 [2/25]  h/J=0.286  [ ORDERED  ]  E_exact=-3.1171  E_vqe=-3.1220  err=0.0049  |M|=0.9746
N=4 [3/25]  h/J=0.371  [ ORDERED  ]  E_exact=-5.1638  E_vqe=-5.1635  err=0.0003  |M|=0.9788
N=6 [3/25]  h/J=0.371  [ ORDERED  ]  E_exact=-3.1926  E_vqe=-3.2066  err=0.0140  |M|=0.9559
N=4 [4/25]  h/J=0.457  [ ORDERED  ]  E_exact=-3.2825  E_vqe=-3.3125  err=0.0300  |M|=0.9327
N=4 [5/25]  h/J=0.543  [ ORDERED  ]  E_exact=-5.2767  E_vqe=-5.2781  err=0.0014  |M|=0.9629
N=6 [4/25]  h/J=0.457  [ ORDERED  ]  E_exact=-3.3847  E_vqe=-3.4400  err=0.0552  |M|=0.9044
N=4 [6/25]  h/J=0.629  [ OR

In [ ]:
# Plot 1: Magnetisation vs h/J
plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (N, data), color in zip(results.items(), colors):
    axes[0].plot(h_values / J, data["magnetisations"],
                 "o-", label=f"N={N}", color=color)
axes[0].axvline(x=1.0, color="gray", linestyle=":", label="h/J = 1")
axes[0].set_xlabel("h / J")
axes[0].set_ylabel("|⟨M⟩|")
axes[0].set_title("Magnetisation vs h/J for increasing N")
axes[0].legend()
axes[0].set_xlim(0, 2)

# Plot 2: VQE error vs system size

for (N, data), color in zip(results.items(), colors):
    axes[1].plot(h_values / J,
                 np.abs(data["vqe_energies"] - data["exact_energies"]),
                 "o-", label=f"N={N}", color=color)
axes[1].axvline(x=1.0, color="gray", linestyle=":")
axes[1].set_xlabel("h / J")
axes[1].set_ylabel("|E_VQE - E_exact|")
axes[1].set_title("VQE error vs system size")
axes[1].legend()
axes[1].set_xlim(0, 3)

plt.tight_layout()
plt.savefig("tfim_finite_size.png", dpi=150)
plt.show()

# Plot 2: VQE data collapse 
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 5))

for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    ax.plot(x_scaled, y_scaled, "o-", label=f"N={N}", color=color)

ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax.set_title("Finite-size scaling collapse (β=1/8, ν=1)")
ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
ax.legend()
ax.set_xlim(-8, 5)
plt.tight_layout()
plt.savefig("tfim_data_collapse.png", dpi=150)
plt.show()


In [ ]:
def exact_magnetisation(N: int, J: float, h: float) -> float:
    epsilon_map   = {4: 1e-6, 6: 1e-4, 8: 1e-4, 10: 1e-4}
    epsilon_local = epsilon_map.get(N, 1e-4)

    H_matrix             = np.array(build_tfim_hamiltonian(N, J, h,
                                     epsilon=epsilon_local).np.array())
    eigenvalues, eigvecs = np.linalg.eigh(H_matrix)
    ground_state         = np.ascontiguousarray(eigvecs[:, 0])
    M_matrix             = np.array(build_magnetisation_op(N).np.array())
    M_val                = ground_state.conj() @ M_matrix @ ground_state
    return abs(float(np.real(M_val)))


def exact_energy_gap(N: int, J: float, h: float) -> float:
    H_matrix    = np.array(build_tfim_hamiltonian(N, J, h).np.array())
    eigenvalues = np.linalg.eigvalsh(H_matrix)
    return float(eigenvalues[1] - eigenvalues[0])

def find_pseudo_critical_point(N, h_values, benchmark):
    """
    Find pseudo-critical point using exact magnetisations.
    Restricts search to h/J in [0.7, 1.3] to avoid boundary artefacts.
    """
    mags   = benchmark[N]["magnetisations"]
    mask   = (h_values / J >= 0.7) & (h_values / J <= 1.3)
    h_sub  = h_values[mask]
    m_sub  = mags[mask]

    dM_dh  = np.gradient(m_sub, h_sub)
    idx    = np.argmin(dM_dh)
    return h_sub[idx]

pseudo_critical = {}
for N in N_values:
    h_c = find_pseudo_critical_point(N, h_values, benchmark)
    pseudo_critical[N] = h_c
    print(f"N={N:2d}  pseudo-critical point: h/J = {h_c:.3f}")

def scaling_form(N, h_c_inf, a):
    return h_c_inf + a * N ** (-1.0 / nu_exp)

N_arr   = np.array(list(pseudo_critical.keys()), dtype=float)
hc_arr  = np.array(list(pseudo_critical.values()))

popt, pcov = curve_fit(scaling_form, N_arr, hc_arr, p0=[1.0, 1.0])
h_c_inf, a = popt
h_c_err    = np.sqrt(pcov[0, 0])

print(f"\nExtrapolated critical point: h_c(∞) = {h_c_inf:.4f} ± {h_c_err:.4f}")
print(f"Exact answer:                h_c(∞) = 1.0000")
print(f"Error:                       {abs(h_c_inf - 1.0)*100:.2f}%")


plt.close('all')
fig = plt.figure(figsize=(18, 10))
fig.suptitle("TFIM Quantum Phase Transition — Full Summary", fontsize=14, fontweight="bold")

# Layout: 2 rows, 3 columns
ax1 = fig.add_subplot(2, 3, 1)   # Ground state energy
ax2 = fig.add_subplot(2, 3, 2)   # Magnetisation
ax3 = fig.add_subplot(2, 3, 3)   # Energy gap
ax4 = fig.add_subplot(2, 3, 4)   # Data collapse
ax5 = fig.add_subplot(2, 3, 5)   # VQE error
ax6 = fig.add_subplot(2, 3, 6)   # Finite-size extrapolation

colors = ["steelblue", "teal", "coral", "purple"]

for (N, data), color in zip(results.items(), colors):
    b         = benchmark[N]
    x         = h_values / J
    x_scaled  = (x - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled  = b["magnetisations"] * (N ** (beta_exp / nu_exp))

    # 1. Ground state energy
    ax1.plot(x, b["energies"],        "-",   color=color, linewidth=2,  label=f"N={N} exact")
    ax1.plot(x, data["vqe_energies"], "o--", color=color, alpha=0.5)

    # 2. Magnetisation
    ax2.plot(x, b["magnetisations"],   "-",   color=color, linewidth=2, label=f"N={N}")
    ax2.plot(x, data["magnetisations"],"o--", color=color, alpha=0.5)

    # 3. Energy gap
    ax3.plot(x, b["gaps"], "-", color=color, linewidth=2, label=f"N={N}")

    # 4. Data collapse (exact)
    ax4.plot(x_scaled, y_scaled, "o-", color=color, label=f"N={N}")

    # 5. VQE error
    ax5.plot(x, np.abs(data["vqe_energies"] - b["energies"]),
             "o-", color=color, label=f"N={N}")

# 6. Finite-size extrapolation
N_fine  = np.linspace(3, 14, 100)
ax6.plot(N_arr, hc_arr, "o", color="steelblue", markersize=8, label="Pseudo-critical points")
ax6.plot(N_fine, scaling_form(N_fine, *popt), "--", color="coral",
         label=f"Fit: $h_c(∞)$ = {h_c_inf:.4f} ± {h_c_err:.4f}")
ax6.axhline(y=1.0, color="gray", linestyle=":", label="Exact $h_c$ = 1.0")
ax6.set_xlabel("N")
ax6.set_ylabel("$h_c(N)$ / J")
ax6.set_title("Finite-size extrapolation")
ax6.legend(fontsize=8)

# Formatting
for ax in [ax1, ax2, ax3, ax5]:
    ax.axvline(x=1.0, color="gray", linestyle=":", alpha=0.7)
    ax.set_xlabel("h / J")

ax4.axvline(x=0, color="gray", linestyle=":", alpha=0.7)
ax4.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax4.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax4.set_xlim(-8, 8)

ax1.set_ylabel("Ground state energy")
ax1.set_title("Ground state energy")
ax1.legend(fontsize=7)

ax2.set_ylabel("|⟨M⟩|")
ax2.set_title("Magnetisation: exact (solid) VQE (dashed)")
ax2.legend(fontsize=8)

ax3.set_ylabel("Δ = E₁ - E₀")
ax3.set_title("Energy gap")
ax3.legend(fontsize=8)

ax4.set_title("Finite-size scaling collapse")
ax4.legend(fontsize=8)

ax5.set_ylabel("|E_VQE - E_exact|")
ax5.set_title("VQE approximation error")
ax5.legend(fontsize=8)

plt.tight_layout()
plt.savefig("tfim_full_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# VISUALISATION OF THE TFIM PHASE DIAGRAM 
# Plotting options
save_figures    = True               # save figures as PNG files
dpi             = 150                # resolution for saved figures
colors          = ["steelblue", "teal", "coral", "purple", "darkgreen", "orange"]  # one per N

# For heatmaps: custom colormap (blue -> white -> red)
cmap_phase      = LinearSegmentedColormap.from_list('phase', ['#1f77b4', 'white', '#d62728'], N=256)

# For phase boundary extrapolation (Figure 6)
nu_fit          = 1.0                # exponent used in fit (exact value = 1)

# Prepare data arrays 
x = h_values / J                      # reduced field

# Figure 1: Magnetisation (exact vs VQE) 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    # Exact magnetisation
    ax1.plot(x, benchmark[N]['magnetisations'], 'o-', color=color,
             linewidth=2, markersize=4, label=f'N = {N}')
    # VQE magnetisation (dashed)
    ax2.plot(x, results[N]['magnetisations'], 'o--', color=color,
             linewidth=2, markersize=4, label=f'N = {N}')

for ax in (ax1, ax2):
    ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5, label='h/J = 1')
    ax.set_xlabel('h / J', fontsize=12)
    ax.set_ylabel(r'$\langle M \rangle$', fontsize=12)  
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
ax1.set_title('Exact magnetisation', fontsize=13)
ax2.set_title('VQE magnetisation', fontsize=13)
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_magnetisation.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 2: VQE energy error
plt.figure(figsize=(8, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    error = np.abs(results[N]['vqe_energies'] - benchmark[N]['energies'])
    plt.semilogy(x, error, 'o-', color=color, linewidth=2, markersize=4, label=f'N = {N}')
plt.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5)
plt.xlabel('h / J', fontsize=12)
plt.ylabel(r'$|E_{\mathrm{VQE}} - E_{\mathrm{exact}}|$', fontsize=12)  
plt.title('VQE energy approximation error', fontsize=13)
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_VQE_error.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 3: Energy gap
plt.figure(figsize=(8, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    plt.plot(x, benchmark[N]['gaps'], 'o-', color=color, linewidth=2, markersize=4, label=f'N = {N}')
plt.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5)
plt.xlabel('h / J', fontsize=12)
plt.ylabel(r'$\Delta = E_1 - E_0$', fontsize=12)  
plt.title('Energy gap closing at the quantum critical point', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_energy_gap.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 4: Finite‑size scaling collapse 
plt.figure(figsize=(7, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    x_scaled = (x - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = benchmark[N]['magnetisations'] * (N ** (beta_exp / nu_exp))
    plt.plot(x_scaled, y_scaled, 'o-', color=color, label=f'N = {N}')
plt.axvline(x=0, color='gray', linestyle='--', linewidth=1.5)
plt.xlabel(r'$(h/J - 1) \, N^{1/\nu}$', fontsize=12)      
plt.ylabel(r'$\langle M \rangle \, N^{\beta/\nu}$', fontsize=12)  
plt.title('Finite‑size scaling collapse (exact data)', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_scaling_collapse.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 5: Heatmaps of magnetisation (exact and VQE) 
M_exact = np.array([benchmark[N]['magnetisations'] for N in N_values])
M_vqe   = np.array([results[N]['magnetisations'] for N in N_values])

fig, (ax5, ax6) = plt.subplots(1, 2, figsize=(12, 5))
extent = [x.min(), x.max(), N_values[0], N_values[-1]]

im1 = ax5.imshow(M_exact, aspect='auto', origin='lower', extent=extent,
                 cmap=cmap_phase, vmin=0, vmax=1)
ax5.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5)
ax5.set_xlabel('h / J', fontsize=12)
ax5.set_ylabel('System size N', fontsize=12)
ax5.set_title('Exact magnetisation', fontsize=13)
cbar1 = plt.colorbar(im1, ax=ax5)
cbar1.set_label(r'$\langle M \rangle$', fontsize=11)   

im2 = ax6.imshow(M_vqe, aspect='auto', origin='lower', extent=extent,
                 cmap=cmap_phase, vmin=0, vmax=1)
ax6.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5)
ax6.set_xlabel('h / J', fontsize=12)
ax6.set_ylabel('System size N', fontsize=12)
ax6.set_title('VQE magnetisation', fontsize=13)
cbar2 = plt.colorbar(im2, ax=ax6)
cbar2.set_label(r'$\langle M \rangle$', fontsize=11)  

plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_phase_heatmap.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 6: Phase boundary shift with N (using your pseudo_critical dictionary) 
# Ensure pseudo_critical is a dict: {N: h_c}
if 'pseudo_critical' in globals():
    N_vals = np.array(sorted(pseudo_critical.keys()))
    hc_vals = np.array([pseudo_critical[N] for N in N_vals])
else:
    # Fallback: compute pseudo_critical from derivative (as you did earlier)
    # This code is already in your notebook; you can copy it here if needed.
    print("pseudo_critical not found; skipping Figure 6.")
    N_vals = np.array([])
    hc_vals = np.array([])

if len(N_vals) > 0:
    plt.figure(figsize=(8, 5))
    plt.errorbar(N_vals, hc_vals, yerr=np.zeros_like(hc_vals), fmt='o', capsize=4,
                 capthick=1, color='teal', label='Pseudo-critical point (derivative min)')
    plt.axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, label='Thermodynamic limit $h_c = 1$')
    plt.xlabel('System size $N$', fontsize=12)
    plt.ylabel('$h_c(N) / J$', fontsize=12)
    plt.title('Finite‑size shift of the phase boundary', fontsize=13)
    plt.legend()
    plt.grid(alpha=0.3)

    # Fit scaling law: h_c(N) = h_c(∞) + a * N^{-1/ν}
    def scaling_law(N, h_inf, a):
        return h_inf + a * N ** (-1.0 / nu_fit)

    popt, pcov = curve_fit(scaling_law, N_vals, hc_vals, p0=[1.0, 1.0])
    N_fine = np.linspace(min(N_vals), max(N_vals), 100)
    plt.plot(N_fine, scaling_law(N_fine, *popt), '--', color='coral',
             label=f'Fit: $h_c(\\infty) = {popt[0]:.3f} \\pm {np.sqrt(pcov[0,0]):.3f}$')
    plt.legend()
    plt.tight_layout()
    if save_figures:
        plt.savefig('TFIM_phase_boundary_vs_N.png', dpi=dpi, bbox_inches='tight')
    plt.show()

In [ ]:
def exact_entanglement_entropy(N: int, J: float, h: float,
                                subsystem_size: int = None) -> float:
    """
    Compute von Neumann entanglement entropy of the ground state.
    Uses epsilon=0 to avoid symmetry-breaking inflation for small N.
    """
    if subsystem_size is None:
        subsystem_size = N // 2

    H_matrix             = np.array(build_tfim_hamiltonian(N, J, h,
                                     epsilon=0.0).np.array())
    eigenvalues, eigvecs = np.linalg.eigh(H_matrix)
    ground_state         = np.ascontiguousarray(eigvecs[:, 0])

    psi             = ground_state.reshape(2**subsystem_size,
                                           2**(N - subsystem_size))
    singular_values = np.linalg.svd(psi, compute_uv=False)
    schmidt_sq      = singular_values**2
    schmidt_sq      = schmidt_sq[schmidt_sq > 1e-12]
    return float(-np.sum(schmidt_sq * np.log(schmidt_sq)))

In [ ]:
for N in N_values:
    M = exact_magnetisation(N, J, h=0.2)
    print(f"N={N:2d}  |M| = {M:.6f}")   # all should be close to 1.0

In [ ]:
# Run sweep for physical characteristics
J = J_meV
h_values = h_values_phys

vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
    N, J, h_values, reps=reps
)

# Plot with physical axis
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(B_values, magnetisations, "o-", color="steelblue", label="VQE")
ax.axvline(x=h_c, color="gray", linestyle=":", label=f"Experimental h_c = {h_c} T")
ax.set_xlabel("Applied field B (Tesla)")
ax.set_ylabel("|⟨M⟩|")
ax.set_title("LiHoF₄ quantum phase transition (TFIM model)")
ax.legend()
plt.tight_layout()
plt.show()